# 06 · CatBoost

Gradient boosting con árboles simétricos (todos los nodos de un mismo nivel usan la misma división) y ordenamiento aleatorio de los datos, lo que reduce el sobreajuste.

Se entrena con junio, se evalúa con septiembre y solo en las localidades habilitadas. Se comparan tres conjuntos de variables (`variables_v2.CONJUNTOS`): `v1` (las del cuaderno 02), `v2` (v1 + tipología, más amenidades, noches mínimas recortadas y estancia mensual) y `v2_geo` (v2 + latitud y longitud).

Además del MdAPE sobre toda la captura se registra `mdape_nuevos`, que es el error en los anuncios que no existían en junio (una simulación del caso real, dado que son anuncios nuevos). Las corridas van al experimento `bogoprice-catboost` de MLflow.

In [1]:
import catboost as cb
import pandas as pd

from bogoprice.experimentos import evaluar, preparar_datos
from bogoprice.tracking import iniciar_mlflow

# Tag de autor de MLflow
AUTOR = "Juan Camilo Piñeros Guzmán"
iniciar_mlflow("bogoprice-catboost")
datos = preparar_datos()
resultados = []


def catb(**p):
    # Valores base: cada prueba cambia solo uno de ellos
    base = {"loss_function": "RMSE", "iterations": 1500, "learning_rate": 0.05, "depth": 8,
            "l2_leaf_reg": 3, "random_strength": 1, "subsample": 0.8,
            "border_count": 254, "rsm": 1.0}
    modelo = cb.CatBoostRegressor(verbose=0, random_seed=0, thread_count=-1,
                                  allow_writing_files=False, **(base | p))
    return modelo, base | p

ModuleNotFoundError: No module named 'bogoprice'

## 1. Conjuntos de variables

La configuración base con los tres conjuntos de variables.

In [ ]:
for version in ["v1", "v2", "v2_geo"]:
    modelo, params = catb()
    m = evaluar(datos, f"catboost_{version}", modelo, version, {"modelo": "CatBoost", **params}, AUTOR)[0]
    resultados.append({"corrida": f"catboost_{version}", "conjunto": version,
                       "parametro": "base", "valor": "-", **m})
    print(f"{version:7s} MdAPE nuevos = {m['mdape_nuevos']:.2f} %")

2026/11/08 18:34:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/11/08 18:34:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run catboost_v1 at: http://localhost:8050/#/experiments/1/runs/db8f7d22edac4de4b0e8c63c0ab8881f
🧪 View experiment at: http://localhost:8050/#/experiments/1
v1      MdAPE nuevos = 20.99 %


2026/11/08 18:34:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/11/08 18:34:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run catboost_v2 at: http://localhost:8050/#/experiments/1/runs/6b5768f72f58473baff8e18cd36ed9f6
🧪 View experiment at: http://localhost:8050/#/experiments/1
v2      MdAPE nuevos = 20.78 %


2026/11/08 18:35:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/11/08 18:35:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run catboost_v2_geo at: http://localhost:8050/#/experiments/1/runs/01a9e08930c74f6ba0f5cd922f9de376
🧪 View experiment at: http://localhost:8050/#/experiments/1
v2_geo  MdAPE nuevos = 18.39 %


## 2. Un hiperparámetro a la vez (v2 con y sin coordenadas)

Se parte de la configuración base y se prueban **4 valores de cada hiperparámetro**, dejando los demás constantes.

| Hiperparámetro | Qué controla | Base | Valores probados |
|---|---|---|---|
| `loss_function` | Función de pérdida que se minimiza | `RMSE` (cuadrática) | `MAE`, `Huber`, `LogCosh`, `Lq` (q = 1,5) |
| `iterations` | Número de árboles | 1500 | 500, 1000, 2500, 4000 |
| `learning_rate` | Cuánto corrige cada árbol | 0,05 | 0,02, 0,03, 0,08, 0,15 |
| `depth` | Profundidad de cada árbol | 8 | 4, 6, 7, 10 |
| `l2_leaf_reg` | Regularización L2 de las hojas | 3 | 1, 5, 10, 30 |
| `random_strength` | Aleatoriedad al elegir divisiones | 1 | 0, 0,5, 2, 5 |
| `subsample` | Fracción de anuncios usada en cada árbol | 0,8 | 0,5, 0,6, 0,7, 1,0 |
| `border_count` | Cortes posibles por variable numérica | 254 | 32, 64, 128, 512 |
| `rsm` | Fracción de variables evaluadas en cada división | 1,0 | 0,3, 0,5, 0,7, 0,9 |

In [ ]:
# Cada parámetro se prueba con 4 valores; los demás quedan en la base
pruebas = {
    "loss_function": ["MAE", "Huber:delta=0.5", "LogCosh", "Lq:q=1.5"],
    "iterations": [500, 1000, 2500, 4000],
    "learning_rate": [0.02, 0.03, 0.08, 0.15],
    "depth": [4, 6, 7, 10],
    "l2_leaf_reg": [1, 5, 10, 30],
    "random_strength": [0, 0.5, 2, 5],
    "subsample": [0.5, 0.6, 0.7, 1.0],
    "border_count": [32, 64, 128, 512],
    "rsm": [0.3, 0.5, 0.7, 0.9],
}

for version in ["v2", "v2_geo"]:
    for parametro, valores in pruebas.items():
        for valor in valores:
            modelo, params = catb(**{parametro: valor})
            nombre = f"catboost_{parametro}{valor}_{version}"
            m = evaluar(datos, nombre, modelo, version,
                        {"modelo": "CatBoost", "parametro_variado": parametro, **params}, AUTOR)[0]
            resultados.append({"corrida": nombre, "conjunto": version, "parametro": parametro,
                               "valor": valor, **m})
        print(f"{version:7s} {parametro} listo")

2026/11/08 18:35:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/11/08 18:35:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run catboost_loss_functionMAE_v2 at: http://localhost:8050/#/experiments/1/runs/940386a91eb34909ad34b1790a1c895a
🧪 View experiment at: http://localhost:8050/#/experiments/1


2026/11/08 18:35:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/11/08 18:35:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run catboost_loss_functionHuber:delta=0.5_v2 at: http://localhost:8050/#/experiments/1/runs/078db9839bba43b698c72f9660220407
🧪 View experiment at: http://localhost:8050/#/experiments/1


... (206 outputs left out)

🏃 View run catboost_rsm0.7_v2_geo at: http://localhost:8050/#/experiments/1/runs/dd853111593b427aa4fc374bb689308b
🧪 View experiment at: http://localhost:8050/#/experiments/1


2026/11/08 18:56:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/11/08 18:56:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run catboost_rsm0.9_v2_geo at: http://localhost:8050/#/experiments/1/runs/71a1a3adacc043d7bd43f36aead03ae3
🧪 View experiment at: http://localhost:8050/#/experiments/1
v2_geo  rsm listo


## Resumen

MdAPE en anuncios nuevos de cada prueba, con y sin coordenadas. La fila `base` es la referencia.

In [ ]:
tabla = pd.DataFrame(resultados)
tabla.pivot_table(index=["parametro", "valor"], columns="conjunto", values="mdape_nuevos").round(2)

conjunto                            v1     v2  v2_geo
parametro       valor                                
base            -                20.99  20.78   18.39
border_count    32                 NaN  20.62   19.06
                64                 NaN  20.78   18.46
                128                NaN  20.78   18.44
                512                NaN  20.78   18.61
depth           4                  NaN  20.46   19.25
                6                  NaN  20.40   18.36
                7                  NaN  20.30   18.14
                10                 NaN  20.79   18.48
iterations      500                NaN  20.36   19.17
                1000               NaN  20.44   18.20
                2500               NaN  20.98   18.40
                4000               NaN  20.92   18.57
l2_leaf_reg     1                  NaN  20.67   18.23
                5                  NaN  20.23   18.34
                10                 NaN  20.37   18.51
                30                 NaN  20.25   18.49
learning_rate   0.02               NaN  20.35   18.75
                0.03               NaN  20.32   18.10
                0.08               NaN  20.35   18.55
                0.15               NaN  20.72   18.41
loss_function   Huber:delta=0.5    NaN  23.83   23.23
                LogCosh            NaN  20.59   18.07
                Lq:q=1.5           NaN  20.62   18.37
                MAE                NaN  20.21   16.99
random_strength 0                  NaN  20.46   18.29
                0.5                NaN  20.32   18.23
                2                  NaN  20.57   18.55
                5                  NaN  20.39   18.50
rsm             0.3                NaN  20.30   18.23
                0.5                NaN  20.50   18.30
                0.7                NaN  20.73   18.28
                0.9                NaN  20.60   18.00
subsample       0.5                NaN  20.28   17.68
                0.6                NaN  20.20   18.48
                0.7                NaN  20.16   17.96
                1                  NaN  20.87   18.36

**Efecto de cada hiperparámetro:** diferencia entre el mejor y el peor de sus 4 valores, y mejora del mejor valor frente a la base (positivo = mejora). Los dos con mayor mejora son los candidatos para explorar en detalle.

In [ ]:
base = tabla[tabla["parametro"] == "base"].set_index("conjunto")["mdape_nuevos"]
pruebas_df = tabla[tabla["parametro"] != "base"]
efecto = (pruebas_df.groupby(["conjunto", "parametro"])["mdape_nuevos"]
          .agg(mejor="min", peor="max")
          .assign(rango=lambda d: d["peor"] - d["mejor"]))
efecto["mejora_vs_base"] = base.reindex(efecto.index.get_level_values("conjunto")).values - efecto["mejor"]
efecto.round(2).sort_values(["conjunto", "mejora_vs_base"], ascending=[True, False])

mejor   peor  rango  mejora_vs_base
conjunto parametro                                           
v2       subsample        20.16  20.87   0.72            0.63
         loss_function    20.21  23.83   3.62            0.57
         l2_leaf_reg      20.23  20.67   0.44            0.56
         rsm              20.30  20.73   0.43            0.49
         depth            20.30  20.79   0.48            0.48
         random_strength  20.32  20.57   0.25            0.47
         learning_rate    20.32  20.72   0.40            0.46
         iterations       20.36  20.98   0.62            0.43
         border_count     20.62  20.78   0.16            0.16
v2_geo   loss_function    16.99  23.23   6.25            1.40
         subsample        17.68  18.48   0.80            0.71
         rsm              18.00  18.30   0.30            0.40
         learning_rate    18.10  18.75   0.65            0.29
         depth            18.14  19.25   1.11            0.25
         iterations       18.20  19.17   0.97            0.19
         l2_leaf_reg      18.23  18.51   0.29            0.16
         random_strength  18.23  18.55   0.32            0.16
         border_count     18.44  19.06   0.62           -0.05

## Mejor Modelo


Entrenar la combinación de los dos mejores parámetros encontrados sobre v2. Los cuales fuero loss_function = "MAE" y subsample = 0.5.

In [ ]:
modelo_final_v2, params_finales_v2 = catb( 
    loss_function="MAE",
    subsample=0.5,
    depth=8,
    iterations=1500
)

NameError: name 'cb' is not defined

In [ ]:
m_final_v2 = evaluar(
    datos,
    "catboost_optimo_mae_sub05_v2",
    modelo_final_v2,
    "v2",
    {"modelo": "CatBoost_Optimo_Final_v2", **params_finales_v2},
    AUTOR
)

In [ ]:
# Registrar en los resultados del cuaderno
resultados.append({
    "corrida": "catboost_optimo_mae_sub05_v2",
    "conjunto": "v2",
    "parametro": "optimo_final_v2",
    "valor": "MAE + subsample 0.5 (v2)",
    **m_final_v2
})

print("=== RESULTADO MODELO FINAL COMBINADO (SOLAMENTE v2) ===")
print(f"MdAPE Anuncios Nuevos: {m_final_v2['mdape_nuevos']:.2f} %")
print(f"MdAPE Septiembre Completo: {m_final_v2['mdape']:.2f} %")
print(f"MAE (COP): ${m_final_v2['mae']:,.0f}")

NameError: name 'resultados' is not defined